Esse notebook analisa a redundância entre personas

In [1]:
import re
import unicodedata

TOKEN = re.compile(r"[^\W_]+", flags=re.UNICODE)

CONTRACTIONS = {
    ("i", "m"): ("i", "am"),
    ("i", "ve"): ("i", "have"),
    ("i", "ll"): ("i", "will"),
    ("i", "d"): ("i", "would"),

    ("you", "re"): ("you", "are"),
    ("you", "ve"): ("you", "have"),
    ("you", "ll"): ("you", "will"),
    ("you", "d"): ("you", "would"),

    ("we", "re"): ("we", "are"),
    ("we", "ve"): ("we", "have"),
    ("we", "ll"): ("we", "will"),
    ("we", "d"): ("we", "would"),

    ("they", "re"): ("they", "are"),
    ("they", "ve"): ("they", "have"),
    ("they", "ll"): ("they", "will"),
    ("they", "d"): ("they", "would"),

    ("he", "s"): ("he", "is"),
    ("he", "d"): ("he", "would"),
    ("she", "s"): ("she", "is"),
    ("she", "d"): ("she", "would"),
    ("it", "s"): ("it", "is"),
    ("it", "d"): ("it", "would"),
    ("that", "s"): ("that", "is"),
    ("there", "s"): ("there", "is"),

    ("isn", "t"): ("is", "not"),
    ("aren", "t"): ("are", "not"),
    ("wasn", "t"): ("was", "not"),
    ("weren", "t"): ("were", "not"),
    ("don", "t"): ("do", "not"),
    ("doesn", "t"): ("does", "not"),
    ("didn", "t"): ("did", "not"),
    ("haven", "t"): ("have", "not"),
    ("hasn", "t"): ("has", "not"),
    ("hadn", "t"): ("had", "not"),
    ("can", "t"): ("can", "not"),
    ("won", "t"): ("will", "not"),
    ("couldn", "t"): ("could", "not"),
    ("wouldn", "t"): ("would", "not"),
    ("shouldn", "t"): ("should", "not"),

    ("would", "ve"): ("would", "have"),
    ("could", "ve"): ("could", "have"),
    ("should", "ve"): ("should", "have"),
}

In [2]:
import re
import unicodedata
from itertools import combinations

import pandas as pd
import requests


# ============================================================
# Configuração
# ============================================================

SERVER = "https://datasets-server.huggingface.co"

DATASETS = {
    "Synthetic Persona Chat": (
        "visual-memory/Synthetic-Persona-Chat-Mapping_1k",
        "description"
    ),
    "PersonaChat revised": (
        "visual-memory/PersonaChat-Mapping_1k",
        "persona_revised"
    ),
    "ConvAI2 revised": (
        "visual-memory/ConvAI2-Mapping_1k",
        "persona_revised"
    ),
}

ID_COLUMN = "persona-id"

STATEMENT_SPLIT = re.compile(r"(?:\r?\n)+|(?<=[.!?])\s+")
TOKEN = re.compile(r"[^\W_]+", flags=re.UNICODE)


# ============================================================
# Normalização
# ============================================================

def normalize(value):
    """Normaliza caixa, Unicode, pontuação, espaços e contrações."""

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    tokens = TOKEN.findall(value)

    expanded = []
    index = 0

    while index < len(tokens):
        pair = tuple(tokens[index:index + 2])
        replacement = CONTRACTIONS.get(pair)

        if replacement is None:
            expanded.append(tokens[index])
            index += 1
        else:
            expanded.extend(replacement)
            index += 2

    return " ".join(expanded)


def signature(text):
    statements = [
        normalize(s)
        for s in STATEMENT_SPLIT.split(text.strip())
        if normalize(s)
    ]

    # Ignora a ordem das frases
    return tuple(sorted(statements))


# ============================================================
# Download
# ============================================================

def load_dataset(repo):
    response = requests.get(
        f"{SERVER}/parquet",
        params={"dataset": repo},
        timeout=180
    )

    response.raise_for_status()

    files = [
        f for f in response.json()["parquet_files"]
        if f["config"] == "default" and f["split"] == "train"
    ]

    return pd.read_parquet(files[0]["url"])


# ============================================================
# Carrega e processa os datasets
# ============================================================

datasets = {}
dataset_texts = {}

for name, (repo, column) in DATASETS.items():

    df = load_dataset(repo)

    personas = {}
    texts = {}

    for persona_id, text in df[[ID_COLUMN, column]].itertuples(
        index=False, name=None
    ):
        if not isinstance(text, str) or not text.strip():
            continue

        persona_id = str(persona_id)

        personas[persona_id] = signature(text)
        texts[persona_id] = text

    datasets[name] = personas
    dataset_texts[name] = texts


# ============================================================
# 1. Diversidade por dataset
# ============================================================

results = []

for name, personas in datasets.items():

    ids = len(personas)
    unique = len(set(personas.values()))

    results.append({
        "Dataset": name,
        "IDs": ids,
        "Personas distintas": unique,
        "Redundantes": ids - unique,
        "Diversidade": unique / ids
    })


# ============================================================
# 2. Três datasets combinados
# ============================================================

all_personas = [
    (f"{dataset}:{persona_id}", sig)
    for dataset, personas in datasets.items()
    for persona_id, sig in personas.items()
]

ids = len(all_personas)
unique = len({sig for _, sig in all_personas})

results.append({
    "Dataset": "Três datasets combinados",
    "IDs": ids,
    "Personas distintas": unique,
    "Redundantes": ids - unique,
    "Diversidade": unique / ids
})


# ============================================================
# 3. Sobreposição entre datasets
# ============================================================

overlap = []

for dataset_1, dataset_2 in combinations(datasets, 2):

    personas_1 = datasets[dataset_1]
    personas_2 = datasets[dataset_2]

    signatures_1 = set(personas_1.values())
    signatures_2 = set(personas_2.values())

    shared = signatures_1 & signatures_2

    ids_1 = sum(sig in shared for sig in personas_1.values())
    ids_2 = sum(sig in shared for sig in personas_2.values())

    overlap.append({
        "Dataset 1": dataset_1,
        "Dataset 2": dataset_2,
        "Personas compartilhadas": len(shared),
        "IDs Dataset 1": ids_1,
        "IDs Dataset 2": ids_2,
    })


# ============================================================
# Tabelas finais
# ============================================================

tabela_diversidade = pd.DataFrame(results)

tabela_sobreposicao = pd.DataFrame(overlap)


display(
    tabela_diversidade.style
    .format({
        "IDs": "{:,.0f}",
        "Personas distintas": "{:,.0f}",
        "Redundantes": "{:,.0f}",
        "Diversidade": "{:.2%}",
    })
    .hide(axis="index")
    .set_caption("Diversidade de personas por dataset")
)


display(
    tabela_sobreposicao.style
    .hide(axis="index")
    .set_caption("Sobreposição exata entre datasets")
)

Dataset,IDs,Personas distintas,Redundantes,Diversidade
Synthetic Persona Chat,"1,994","1,565",429,78.49%
PersonaChat revised,"1,991","1,534",457,77.05%
ConvAI2 revised,"1,712","1,474",238,86.10%
Três datasets combinados,"5,697","3,918","1,779",68.77%


Dataset 1,Dataset 2,Personas compartilhadas,IDs Dataset 1,IDs Dataset 2
Synthetic Persona Chat,PersonaChat revised,0,0,0
Synthetic Persona Chat,ConvAI2 revised,0,0,0
PersonaChat revised,ConvAI2 revised,655,946,803


Sobreposição de cada Dataset

In [3]:
from collections import defaultdict

def print_redundantes(n=3):
    for dataset, personas in datasets.items():

        grupos = defaultdict(list)

        for persona_id, sig in personas.items():
            grupos[sig].append(persona_id)

        duplicados = [
            ids
            for ids in grupos.values()
            if len(ids) > 1
        ]

        print(f"\n{'='*80}")
        print(dataset)
        print(f"Grupos redundantes: {len(duplicados)}")
        print("="*80)

        for i, ids in enumerate(duplicados[:n], 1):

            print(f"\nExemplo {i} — IDs: {ids}")

            for persona_id in ids:
                print(f"\nID {persona_id}:")
                print(dataset_texts[dataset][persona_id])

            print("-" * 80)

In [4]:
print_redundantes(n=3)


Synthetic Persona Chat
Grupos redundantes: 318

Exemplo 1 — IDs: ['e439fbe3ab6acfa42d8d592ae76af0a15866d5a6f547a63ab5e1581ea3cf053a', '6889fadac76b5cacb77f00bafacc4985f507996b151811d3bc0d4ba539bd60e4']

ID e439fbe3ab6acfa42d8d592ae76af0a15866d5a6f547a63ab5e1581ea3cf053a:
My friend once bought me a car.
My favorite season is winter.
I take vitamin c when i have a cold.
I do not eat bread.
I am disabled and cannot walk.

ID 6889fadac76b5cacb77f00bafacc4985f507996b151811d3bc0d4ba539bd60e4:
I take vitamin c when i have a cold.
My favorite season is winter.
My friend once bought me a car.
I do not eat bread.
I am disabled and cannot walk.
--------------------------------------------------------------------------------

Exemplo 2 — IDs: ['944aa9fab6ad074b56f86729b8eaf78c2ceff2fd0b100da932a83ccd478f5b5e', '4e6afbee6027af7810db5b24ede19e4c2eb9b456eb02b4e66adf535a30acdfbc']

ID 944aa9fab6ad074b56f86729b8eaf78c2ceff2fd0b100da932a83ccd478f5b5e:
I drink a lot of tea.
I hope to one day be a publis

Sobreposição entre datasets:

In [5]:
def print_sobreposicoes(n=3):

    for dataset_1, dataset_2 in combinations(datasets, 2):

        personas_1 = datasets[dataset_1]
        personas_2 = datasets[dataset_2]

        # assinatura -> IDs
        grupos_1 = defaultdict(list)
        grupos_2 = defaultdict(list)

        for persona_id, sig in personas_1.items():
            grupos_1[sig].append(persona_id)

        for persona_id, sig in personas_2.items():
            grupos_2[sig].append(persona_id)

        shared = set(grupos_1) & set(grupos_2)

        print(f"\n{'='*80}")
        print(f"{dataset_1}  ×  {dataset_2}")
        print(f"Personas compartilhadas: {len(shared)}")
        print("="*80)

        for i, sig in enumerate(list(shared)[:n], 1):

            ids_1 = grupos_1[sig]
            ids_2 = grupos_2[sig]

            print(f"\nExemplo {i}")

            print(f"\n{dataset_1} — IDs {ids_1}")
            for persona_id in ids_1:
                print(dataset_texts[dataset_1][persona_id])

            print(f"\n{dataset_2} — IDs {ids_2}")
            for persona_id in ids_2:
                print(dataset_texts[dataset_2][persona_id])

            print("-" * 80)

In [6]:
print_sobreposicoes(n=3)


Synthetic Persona Chat  ×  PersonaChat revised
Personas compartilhadas: 0

Synthetic Persona Chat  ×  ConvAI2 revised
Personas compartilhadas: 0

PersonaChat revised  ×  ConvAI2 revised
Personas compartilhadas: 655

Exemplo 1

PersonaChat revised — IDs ['0eed608458966f9fbd86042f476a1b43c40d487a8bd5921f716e810d0642fc5f', '406fdc3b1507645d0c43b7d3277221fee4bbb425ddc1f3380c3f145b8cbcbb66']
i would never even look at meat , yet alone eat it. i enjoy baking. i just started my current position. i am on the computer a lot. i like to go to concerts.
i like to go to concerts. i would never even look at meat , yet alone eat it. i am on the computer a lot. i just started my current position. i enjoy baking.

ConvAI2 revised — IDs ['afc340919608ca7c770e816f8450f26ce3081a8c9dd2873ea904ed7c94df77ab', 'eaf77a4b7b09560b006eae9ba37532d18ee4b46a6795208e8ba7e86f511d3985']
i am on the computer a lot. i enjoy baking. i just started my current position. i like to go to concerts. i would never even look at 